# Distant Listening Corpus notes → BOPP

Same recipe as the harmony notebook, applied to the `.notes.tsv` facet (note events): load `distant_listening_corpus.datapackage.json` with **dimcat**, derive a BOPP payload schema from the descriptor, represent one example piece as a BOPP `Annotation` (extents in quarter notes), and export it to every BOPP format: **msgpack**, **JSON**, and **CSV**.

In [1]:
import json
from pathlib import Path

from dimcat import Dataset

ROOT = Path.cwd()  # this notebook lives in the repo root, next to the descriptor
ds = Dataset()
ds.load(str(ROOT / "distant_listening_corpus.datapackage.json"))
pkg = ds.inputs.get_package("distant_listening_corpus")
print(pkg.resource_names)

/home/laser/git/bopp/.venv/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


['distant_listening_corpus.measures', 'distant_listening_corpus.notes', 'distant_listening_corpus.expanded', 'distant_listening_corpus.chords', 'distant_listening_corpus.metadata']


In [2]:
# Payload schema <- every descriptor column except the row identifiers
# (corpus, piece, i) and the two extent source columns.
descriptor = json.loads((ROOT / "distant_listening_corpus.datapackage.json").read_text())
notes_res = next(r for r in descriptor["resources"] if r["name"] == "distant_listening_corpus.notes")
fields = notes_res["schema"]["fields"]
INDEX = {"corpus", "piece", "i"}
EXTENT_SRC = {"quarterbeats_all_endings", "duration_qb"}
payload_fields = [f for f in fields if f["name"] not in INDEX | EXTENT_SRC]
print("payload columns:", [f["name"] for f in payload_fields])
for f in payload_fields:
    print(f"- {f['name']}: {f['type']} — {f.get('title', '')}: {f.get('description', '')}")

payload columns: ['mc', 'mn', 'quarterbeats', 'mc_onset', 'mn_onset', 'timesig', 'staff', 'voice', 'duration', 'gracenote', 'nominal_duration', 'scalar', 'tied', 'tpc', 'midi', 'name', 'octave', 'chord_id', 'tremolo', 'volta', 'tuning']
- mc: integer — Measure Count: Running count of encoded <Measure> tags which do not necessarily express a full measure (e.g. in case of an anacrusis).
- mn: string — Measure Number: Measure number as printed in the score, computed from mc, dont_count and numbering_offset.
- quarterbeats: string — Offset from Beginning (leaving out alternative endings): Distance from the piece's beginning. By default, only second endings are taken into account to reflect the proportions of a simply playthrough without repeats.
- mc_onset: string — Offset within Encoded Measure: Distance of an event from the beginning of the <Measure> tag.
- mn_onset: string — Offset within Logical Measure: Distance from the beginning of the logical measure. Relevant, for example, to comp

In [3]:
notes = pkg.get_resource_by_name("distant_listening_corpus.notes")
df = notes.df
print(type(df), df.shape)
print("index levels:", df.index.names)
df.head(3)

<class 'pandas.core.frame.DataFrame'> (1688754, 23)
index levels: ['corpus', 'piece', 'i']


mc mn quarterbeats quarterbeats_all_endings  \
corpus piece        i                                                
ABC    n01op18-1_01 0   1  1            0                        0   
                    1   1  1            0                        0   
                    2   1  1            0                        0   

                       duration_qb mc_onset mn_onset timesig  staff  voice  \
corpus piece        i                                                        
ABC    n01op18-1_01 0          1.0        0        0     3/4      3      1   
                    1          1.0        0        0     3/4      4      1   
                    2          1.0        0        0     3/4      1      1   

                       ... scalar tied tpc midi  name  octave  chord_id  \
corpus piece        i  ...                                                
ABC    n01op18-1_01 0  ...      1    1  -1   53    F3       3        12   
                    1  ...      1    1  -1   53    F3       3        18   
                    2  ...      1    1  -1   65    F4       4         0   

                      tremolo  volta  tuning  
corpus piece        i                         
ABC    n01op18-1_01 0    <NA>   <NA>    <NA>  
                    1    <NA>   <NA>    <NA>  
                    2    <NA>   <NA>    <NA>  

[3 rows x 23 columns]

In [4]:
# Same example piece as the harmony notebook, for comparability.
corpus, piece = "ABC", "n01op18-1_01"
piece_df = df.loc[(corpus, piece)]
print(piece_df.shape)
print("quarterbeats_all_endings element type:", type(piece_df["quarterbeats_all_endings"].iloc[0]))
print("duration element type:", type(piece_df["duration"].iloc[0]))

# Extents: quarterbeats_all_endings fractions -> float "start", duration_qb -> "duration"
starts = [float(v) for v in piece_df["quarterbeats_all_endings"].tolist()]
durations = [float(v) for v in piece_df["duration_qb"].tolist()]
print("starts[:5]:", starts[:5])
print("durations[:5]:", durations[:5])

(4722, 23)
quarterbeats_all_endings element type: <class 'fractions.Fraction'>
duration element type: <class 'fractions.Fraction'>
starts[:5]: [0.0, 0.0, 0.0, 0.0, 1.0]
durations[:5]: [1.0, 1.0, 1.0, 1.0, 0.5]


## BOPP schemata

- **Payload** `schemas/v1/payload/score_note.json` (`payload_type: score_note`): one parallel array per descriptor column above (21 arrays). Fraction-typed columns (`quarterbeats`, `mc_onset`, `mn_onset`, `duration`, `nominal_duration`, `scalar`) are stored as floats — the descriptor declares fraction strings, but dimcat materializes `Fraction` objects, so float conversion keeps them numeric (same choice as for the extents).
- **Extent**: reuses `quarter_interval` (`start` + `duration` in quarter notes) — no new extent type needed.

In [5]:
import math
from fractions import Fraction

import msgspec
import numpy as np
import pandas as pd

from bopp.io import decode_arrow
from bopp.models.v1.annotation import Annotation

PAYLOAD_COLUMNS = [f["name"] for f in payload_fields]


def _scalar(v):
    """Normalize one pandas cell to msgpack/Arrow-friendly plain Python."""
    if v is None or v is pd.NA:
        return None
    if isinstance(v, float) and math.isnan(v):
        return None
    if isinstance(v, tuple):
        return [_scalar(x) for x in v]
    if isinstance(v, np.integer):
        return int(v)
    if isinstance(v, np.floating):
        return float(v)
    if isinstance(v, np.bool_):
        return bool(v)
    if isinstance(v, Fraction):
        return float(v)
    return v


payload = {"payload_type": "score_note"}
for col in PAYLOAD_COLUMNS:
    payload[col] = [_scalar(v) for v in piece_df[col].tolist()]

annotation = msgspec.convert(
    {
        "media_id": f"dlc:{corpus}/{piece}",
        "bopp_version": "1.0.0",
        "extent": {"extent_type": "quarter_interval", "start": starts, "duration": durations},
        "payload": payload,
    },
    type=Annotation,
    dec_hook=decode_arrow,
)
print(type(annotation.payload).__name__, "|", type(annotation.extent).__name__)
print("n observations:", len(annotation.extent.start))

ScoreNotePayload | QuarterIntervalExtent
n observations: 4722


In [6]:
from bopp.util import to_dataframe

bdf = to_dataframe(annotation)
print(bdf.shape)
bdf[["extent:quarter_interval:start", "extent:quarter_interval:duration",
     "payload:score_note:mc", "payload:score_note:name",
     "payload:score_note:midi", "payload:score_note:tpc"]].head(8)

(4722, 23)


,extent:quarter_interval:start,extent:quarter_interval:duration,payload:score_note:mc,payload:score_note:name,payload:score_note:midi,payload:score_note:tpc
0,0.0,1.0,1,F3,53,-1
1,0.0,1.0,1,F3,53,-1
2,0.0,1.0,1,F4,65,-1
3,0.0,1.0,1,F4,65,-1
4,1.0,0.5,1,F3,53,-1
5,1.0,0.5,1,F3,53,-1
6,1.0,0.5,1,F4,65,-1
7,1.0,0.5,1,F4,65,-1


In [7]:
from bopp.io import encode_arrow, load_bopp_msgpack, save_bopp_msgpack

save_bopp_msgpack(annotation, str(ROOT / "dlc_example_notes.bopp.msgpack"))

reloaded = load_bopp_msgpack(str(ROOT / "dlc_example_notes.bopp.msgpack"))
assert msgspec.msgpack.encode(reloaded, enc_hook=encode_arrow) == msgspec.msgpack.encode(
    annotation, enc_hook=encode_arrow
)
print("msgpack round-trip byte-identical;",
      f"first note: {reloaded.payload.name.to_pylist()[0]!r} (MIDI {reloaded.payload.midi.to_pylist()[0]})")

Successfully saved to /home/laser/git/bopp/dlc_example_notes.bopp.msgpack (461369 bytes)
msgpack round-trip byte-identical; first note: 'F3' (MIDI 53)


In [8]:
from bopp.io import load_bopp_json, save_bopp_json

save_bopp_json(annotation, str(ROOT / "dlc_example_notes.bopp.json"))
from_json = load_bopp_json(str(ROOT / "dlc_example_notes.bopp.json"))
assert msgspec.msgpack.encode(from_json, enc_hook=encode_arrow) == msgspec.msgpack.encode(
    annotation, enc_hook=encode_arrow
)
print("JSON round-trip byte-identical")

Successfully saved to /home/laser/git/bopp/dlc_example_notes.bopp.json (493463 bytes)
JSON round-trip byte-identical


In [9]:
from bopp.io import load_bopp_csv, to_csv

to_csv(annotation, str(ROOT / "dlc_example_notes.bopp.csv"))
from_csv = load_bopp_csv(str(ROOT / "dlc_example_notes.bopp.csv"))

d1, d2 = to_dataframe(annotation), to_dataframe(from_csv)
assert d1.shape == d2.shape and list(d1.columns) == list(d2.columns)


def _cells(s):
    out = []
    for x in s.tolist():
        x = x.as_py() if hasattr(x, "as_py") else (x.tolist() if isinstance(x, np.ndarray) else x)
        out.append(None if (x is None or (isinstance(x, float) and math.isnan(x))) else x)
    return out


def _canon(v):
    # CSV is untyped: nullable ints come back as floats, digit strings as ints.
    if v is None or isinstance(v, bool):
        return v
    if isinstance(v, (int, float)):
        return float(v)
    if isinstance(v, str):
        try:
            return float(v)
        except ValueError:
            return v
    return v


for col in d1.columns:
    assert [_canon(v) for v in _cells(d1[col])] == [_canon(v) for v in _cells(d2[col])], col
print(f"CSV round-trip OK: {d2.shape[0]} rows x {d2.shape[1]} columns, all values equal")

CSV round-trip OK: 4722 rows x 23 columns, all values equal


## Notes / known CSV caveats

- **msgpack and JSON round-trips are byte-identical** (verified above by re-encoding).
- **CSV is untyped**, so column *types* do not survive: nullable integer columns (e.g. `tied`, `volta`) come back as floats, and all-digit strings (e.g. `mn`, `figbass`-style codes) may come back as ints — *values* are preserved. This is inherent to CSV, not BOPP.
- Two small `bopp/io.py` fixes were needed to make CSV work at all and are now in the library: `to_csv` serializes list-like cells as valid Python literals (numpy used to write `[0 4 1]`), and `read_bopp_csv` maps empty fields back to nulls instead of `NaN`.